# Stock Agent Setup Guide

## Prerequisites
- **Ollama**: Ensure Ollama is running locally on `http://localhost:11434`
- **Python 3.12+**: Required for the virtual environment
- **Virtual Environment**: Created at `c:\OllamaLearning\stock_agent_env`

## To Run This Notebook

### Step 1: Activate Virtual Environment (PowerShell)
```powershell
cd c:\OllamaLearning
.\stock_agent_env\Scripts\Activate.ps1
```

### Step 2: Launch Jupyter
```powershell
jupyter notebook
```

### Step 3: Select Kernel
- Open this notebook in Jupyter
- Click `Kernel` → `Change kernel` → Select `Stock Agent Env`

## Required Packages
All dependencies have been installed in the virtual environment:

- **LangChain**: `langchain`, `langchain-core`, `langchain-community`, `langchain-ollama`
- **Data Fetching**: `yfinance`, `beautifulsoup4`, `requests`, `selenium`
- **Visualization**: `matplotlib`, `pandas`
- **Jupyter**: `jupyter`, `ipykernel`, `notebook`

## Quick Install (if needed)
If packages are missing, run in the activated environment:
```bash
pip install langchain langchain-core langchain-community langchain-ollama yfinance beautifulsoup4 requests selenium matplotlib pandas jupyter
```

# LangChain Stock Agent with Upstox Integration

This notebook creates an AI agent that can fetch stock information from upstox.com and plot graphs using LangChain.

In [29]:
import json
import re
import requests
from bs4 import BeautifulSoup
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from typing import Any
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent
import warnings
warnings.filterwarnings('ignore')

In [30]:
# Define Tool 1: Search for Indian stocks by name
@tool
def search_indian_stock(stock_name: str) -> dict:
    """
    Search for Indian stocks by company name and return NSE symbol.
    Searches through NSE (National Stock Exchange) stocks.
    Returns stock symbol, company name, and other details.
    Example: 'Tata' -> 'TCS.NS', 'Reliance' -> 'RELIANCE.NS'
    """
    try:
        import yfinance as yf
        
        # Comprehensive Indian stock database (NSE stocks)
        # Maps company names to their NSE symbols
        indian_stocks_db = {
            # IT & Software
            'tcs': 'TCS.NS',
            'infosys': 'INFY.NS',
            'wipro': 'WIPRO.NS',
            'hcl': 'HCLTECH.NS',
            'tech mahindra': 'TECHM.NS',
            'mindtree': 'MINDTREE.NS',
            'mphasis': 'MPHASIS.NS',
            'ltimindtree': 'LTIM.NS',
            
            # Financial Services
            'icici bank': 'ICICIBANK.NS',
            'hdfc bank': 'HDFCBANK.NS',
            'sbi': 'SBIN.NS',
            'axis bank': 'AXISBANK.NS',
            'kotak bank': 'KOTAKBANK.NS',
            'hdfc': 'HDFC.NS',
            'icici': 'ICICIPRULI.NS',
            'indusind': 'INDUSINDBK.NS',
            
            # Energy & Oil
            'reliance': 'RELIANCE.NS',
            'bharati': 'BHARTIARTL.NS',
            'oil': 'ONGC.NS',
            'power grid': 'POWERGRID.NS',
            
            # Automobiles
            'maruti': 'MARUTI.NS',
            'hero': 'HEROMOTOCO.NS',
            'bajaj': 'BAJAJFINSV.NS',
            'tvs': 'TVS.NS',
            
            # Pharma
            'cipla': 'CIPLA.NS',
            'dr reddy': 'DRREDDY.NS',
            'sun pharma': 'SUNPHARMA.NS',
            'lupin': 'LUPIN.NS',
            'aurobindo': 'AUROPHARMA.NS',
            
            # Consumer & FMCG
            'hindustan unilever': 'HINDUNILVR.NS',
            'nestlé': 'NESTLEIND.NS',
            'britannia': 'BRITANNIA.NS',
            'ift': 'ITC.NS',
            'colgate': 'COLPAL.NS',
            
            # Metals & Mining
            'tata steel': 'TATASTEEL.NS',
            'hindalco': 'HINDALCO.NS',
            'national aluminium': 'NATIONALAL.NS',
            'sail': 'SAIL.NS',
            
            # Infrastructure & Construction
            'larsen toubro': 'LT.NS',
            'dlf': 'DLF.NS',
            'hyundai': 'HYUNDAI.NS',
            
            # Media
            'zee': 'ZEE.NS',
            'sony': 'SONYLDH.NS',
        }
        
        # Search for matching stock
        search_lower = stock_name.lower().strip()
        
        # Exact match
        if search_lower in indian_stocks_db:
            symbol = indian_stocks_db[search_lower]
            return {
                'success': True,
                'query': stock_name,
                'symbol': symbol,
                'company_name': stock_name.title(),
                'exchange': 'NSE',
                'note': 'Exact match found'
            }
        
        # Partial match
        matches = []
        for key, symbol in indian_stocks_db.items():
            if search_lower in key or key in search_lower:
                matches.append({'name': key, 'symbol': symbol})
        
        if matches:
            # Return the best match
            best_match = matches[0]
            return {
                'success': True,
                'query': stock_name,
                'symbol': best_match['symbol'],
                'company_name': best_match['name'].title(),
                'exchange': 'NSE',
                'matches': len(matches),
                'note': 'Partial match found'
            }
        
        # No match found
        return {
            'success': False,
            'query': stock_name,
            'error': f'Stock "{stock_name}" not found in Indian stock database',
            'suggestion': 'Try searching with company name like: TCS, Infosys, Wipro, Reliance, HDFC, ICICI, Maruti, etc.'
        }
    
    except Exception as e:
        return {'success': False, 'error': f'Error searching stock: {str(e)}'}


# Define Tool 1B: Scrape stock data from Upstox
@tool
def scrape_upstox_stock_data(stock_symbol: str) -> dict:
    """
    Scrape stock data from upstox.com for a given stock symbol.
    Returns basic stock information including price, change, and other metrics.
    """
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        
        # Upstox stock search URL
        url = f"https://upstox.com/stocks/{stock_symbol.lower()}"
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Extract stock data from page
        stock_data = {
            'symbol': stock_symbol.upper(),
            'url': url,
            'source': 'upstox.com'
        }
        
        # Try to extract price information
        price_text = soup.find('span', class_=re.compile('.*price.*', re.I))
        if price_text:
            stock_data['current_price'] = price_text.get_text(strip=True)
        
        # Extract other relevant data
        for text in soup.find_all(['span', 'p', 'div']):
            content = text.get_text(strip=True)
            if any(keyword in content.lower() for keyword in ['change', 'high', 'low', 'volume']):
                stock_data['metrics'] = content
        
        return stock_data
    except requests.exceptions.RequestException as e:
        return {'error': f'Failed to scrape Upstox for {stock_symbol}: {str(e)}',
                'suggestion': f'Using alternative data source (Yahoo Finance) for {stock_symbol}'}
    except Exception as e:
        return {'error': f'Error processing data: {str(e)}'}


In [31]:
# Define Tool 2: Fetch historical stock data
@tool
def fetch_stock_historical_data(stock_symbol: str, period: str = "1y") -> dict:
    """
    Fetch historical stock data using Yahoo Finance.
    Periods: 1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max
    Returns historical data with dates and prices.
    """
    try:
        ticker = yf.Ticker(stock_symbol)
        df = ticker.history(period=period)
        
        if df.empty:
            return {'error': f'No data found for {stock_symbol}'}
        
        # Convert to dict for JSON serialization
        result = {
            'symbol': stock_symbol.upper(),
            'period': period,
            'data_points': len(df),
            'start_date': str(df.index[0].date()),
            'end_date': str(df.index[-1].date()),
            'latest_close': float(df['Close'].iloc[-1]),
            'latest_open': float(df['Open'].iloc[-1]),
            'latest_high': float(df['High'].iloc[-1]),
            'latest_low': float(df['Low'].iloc[-1]),
            'latest_volume': int(df['Volume'].iloc[-1]),
            'price_change': float(df['Close'].iloc[-1] - df['Close'].iloc[0]),
            'percent_change': float((df['Close'].iloc[-1] - df['Close'].iloc[0]) / df['Close'].iloc[0] * 100),
            'avg_volume': float(df['Volume'].mean()),
            'data_df': df.to_json()
        }
        return result
    except Exception as e:
        return {'error': f'Error fetching data for {stock_symbol}: {str(e)}'}

In [32]:
# Define Tool 3: Plot stock graph
@tool
def plot_stock_graph(stock_symbol: str, period: str = "1y", include_volume: bool = False) -> str:
    """
    Plot a stock price graph for the specified stock and time period.
    Shows closing price trend over time with optional volume data.
    """
    try:
        ticker = yf.Ticker(stock_symbol)
        df = ticker.history(period=period)
        
        if df.empty:
            return f'No data available to plot for {stock_symbol}'
        
        # Create figure with subplots
        if include_volume:
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})
        else:
            fig, ax1 = plt.subplots(figsize=(14, 6))
        
        # Plot closing price
        ax1.plot(df.index, df['Close'], linewidth=2, color='#1f77b4', label='Close Price')
        ax1.fill_between(df.index, df['Close'], alpha=0.2, color='#1f77b4')
        ax1.set_title(f'{stock_symbol.upper()} Stock Price - {period}', fontsize=16, fontweight='bold')
        ax1.set_ylabel('Price ($)', fontsize=12)
        ax1.grid(True, alpha=0.3)
        ax1.legend(loc='best')
        
        # Add some statistics
        latest_price = df['Close'].iloc[-1]
        min_price = df['Close'].min()
        max_price = df['Close'].max()
        stats_text = f'Latest: ${latest_price:.2f} | Min: ${min_price:.2f} | Max: ${max_price:.2f}'
        ax1.text(0.02, 0.95, stats_text, transform=ax1.transAxes, 
                fontsize=10, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        # Plot volume if requested
        if include_volume:
            ax2.bar(df.index, df['Volume'], color='#ff7f0e', alpha=0.6)
            ax2.set_ylabel('Volume', fontsize=12)
            ax2.set_xlabel('Date', fontsize=12)
            ax2.grid(True, alpha=0.3)
        else:
            ax1.set_xlabel('Date', fontsize=12)
        
        plt.tight_layout()
        
        # Save the plot
        filename = f'{stock_symbol.upper()}_stock_graph_{datetime.now().strftime("%Y%m%d_%H%M%S")}.png'
        plt.savefig(filename, dpi=100, bbox_inches='tight')
        plt.show()
        
        return f'Successfully plotted graph for {stock_symbol} and saved as {filename}'
    except Exception as e:
        return f'Error plotting graph for {stock_symbol}: {str(e)}'

In [33]:
# Define Tool 4: Compare multiple stocks
@tool
def compare_stocks(stock_symbols: str, period: str = "6mo") -> dict:
    """
    Compare multiple stocks' performance over a period.
    stock_symbols should be comma-separated (e.g., "AAPL,GOOGL,MSFT")
    """
    try:
        symbols = [s.strip().upper() for s in stock_symbols.split(',')]
        comparison_data = {}
        
        for symbol in symbols:
            try:
                ticker = yf.Ticker(symbol)
                df = ticker.history(period=period)
                
                if not df.empty:
                    comparison_data[symbol] = {
                        'latest_price': float(df['Close'].iloc[-1]),
                        'price_change': float(df['Close'].iloc[-1] - df['Close'].iloc[0]),
                        'percent_change': float((df['Close'].iloc[-1] - df['Close'].iloc[0]) / df['Close'].iloc[0] * 100),
                        '52_week_high': float(df['High'].max()),
                        '52_week_low': float(df['Low'].min()),
                    }
            except Exception as e:
                comparison_data[symbol] = {'error': str(e)}
        
        return {'comparison': comparison_data, 'period': period}
    except Exception as e:
        return {'error': f'Error comparing stocks: {str(e)}'}

In [34]:
# Define Tool 5: Plot comparison graph
@tool
def plot_comparison_graph(stock_symbols: str, period: str = "6mo") -> str:
    """
    Plot a comparison graph for multiple stocks.
    stock_symbols should be comma-separated (e.g., "AAPL,GOOGL,MSFT")
    """
    try:
        symbols = [s.strip().upper() for s in stock_symbols.split(',')]
        
        fig, ax = plt.subplots(figsize=(14, 7))
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
        
        for idx, symbol in enumerate(symbols):
            try:
                ticker = yf.Ticker(symbol)
                df = ticker.history(period=period)
                
                if not df.empty:
                    # Normalize prices to percentage change from start
                    normalized = (df['Close'] / df['Close'].iloc[0] - 1) * 100
                    color = colors[idx % len(colors)]
                    ax.plot(df.index, normalized, linewidth=2, label=symbol, color=color)
            except:
                pass
        
        ax.set_title(f'Stock Price Comparison - {period}', fontsize=16, fontweight='bold')
        ax.set_ylabel('Percent Change (%)', fontsize=12)
        ax.set_xlabel('Date', fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.legend(loc='best', fontsize=11)
        ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
        
        plt.tight_layout()
        
        filename = f'stock_comparison_{datetime.now().strftime("%Y%m%d_%H%M%S")}.png'
        plt.savefig(filename, dpi=100, bbox_inches='tight')
        plt.show()
        
        return f'Successfully plotted comparison graph for {", ".join(symbols)} and saved as {filename}'
    except Exception as e:
        return f'Error plotting comparison graph: {str(e)}'

In [35]:
# Define Tool 6: Scrape web pages for stock information
@tool
def scrape_stock_web_info(company_name: str, stock_symbol: str = "", max_pages: int = 5) -> dict:
    """
    Scrape stock information from web sources.
    Searches for the company, scrapes top web pages, and returns aggregated information.
    company_name: Full company name (e.g., "Tata Consultancy Services")
    stock_symbol: Stock symbol (e.g., "TCS.NS")
    max_pages: Number of pages to scrape (default: 5)
    """
    try:
        from urllib.parse import quote
        import time
        
        search_query = f"{company_name} stock price news"
        print(f"   🌐 Searching for: {search_query}")
        
        scraped_data = {
            'company': company_name,
            'symbol': stock_symbol,
            'search_query': search_query,
            'pages_scraped': 0,
            'content': [],
            'sources': [],
            'method': 'web_scraping'
        }
        
        # List of financial news and stock sites to scrape
        search_sites = [
            f"https://www.google.com/search?q={quote(search_query)}",
            f"https://www.investopedia.com/search?q={quote(company_name)}",
            f"https://www.cnbc.com/search/?query={quote(company_name)}",
            f"https://economictimes.indiatimes.com/search.cms?query={quote(company_name)}",
            f"https://business.ndtv.com/search?query={quote(company_name)}"
        ]
        
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1'
        }
        
        # Try multiple sources
        for url in search_sites[:max_pages]:
            try:
                response = requests.get(url, headers=headers, timeout=10)
                response.raise_for_status()
                soup = BeautifulSoup(response.content, 'html.parser')
                
                # Remove script and style elements
                for script in soup(['script', 'style']):
                    script.decompose()
                
                # Extract text
                text = soup.get_text(separator=' ', strip=True)
                text = ' '.join(text.split())
                
                # Extract title
                title = soup.title.string if soup.title else 'Unknown'
                
                # Filter content for relevance
                if any(keyword in text.lower() for keyword in ['stock', 'price', 'market', 'trade', 'financial', company_name.lower()]):
                    # Get first 1500 chars of relevant content
                    if len(text) > 500:
                        text = text[:1500]
                    
                    if len(text) > 200:  # Only include if substantial
                        scraped_data['content'].append(text)
                        scraped_data['sources'].append({
                            'url': url[:80],
                            'title': title[:100]
                        })
                        scraped_data['pages_scraped'] += 1
                
                time.sleep(1)  # Be respectful
                
            except requests.exceptions.Timeout:
                print(f"      ⏱️  Timeout: {url[:50]}")
            except requests.exceptions.RequestException as e:
                print(f"      ⚠️  Network error on {url[:50]}: {type(e).__name__}")
            except Exception as e:
                print(f"      ⚠️  Error parsing {url[:50]}: {str(e)[:50]}")
                pass
        
        # If no real web data scraped, create informative placeholder
        if not scraped_data['content']:
            scraped_data['content'] = [
                f"Stock Profile: {company_name} ({stock_symbol})\n"
                f"Note: Real-time web scraping limited due to website restrictions. "
                f"Data fetched from Yahoo Finance provides historical price information. "
                f"For current market sentiment, please check financial news websites directly."
            ]
            scraped_data['pages_scraped'] = 1
            scraped_data['method'] = 'fallback_info'
        
        print(f"      ✓ Processed {scraped_data['pages_scraped']} sources\n")
        return scraped_data
        
    except Exception as e:
        return {
            'company': company_name,
            'symbol': stock_symbol,
            'error': f'Web scraping limitation: {str(e)[:50]}',
            'content': [f"Information for {company_name} ({stock_symbol}). Direct web scraping encountered restrictions."],
            'pages_scraped': 1,
            'method': 'error_fallback'
        }


In [36]:
# Create LangChain tools list
tools = [
    search_indian_stock,
    scrape_upstox_stock_data,
    scrape_stock_web_info,
    fetch_stock_historical_data,
    plot_stock_graph,
    compare_stocks,
    plot_comparison_graph
]

# Initialize LLM with settings for tool use
# Mistral works better with explicit prompting
llm = ChatOllama(
    model="mistral",
    temperature=0,
    base_url="http://localhost:11434"
)

# For Mistral, we don't use bind_tools - instead we use the agent directly
# and let it make decisions about which tools to call
from langgraph.prebuilt import create_react_agent

# Create ReAct agent - this handles tool invocation properly
agent = create_react_agent(llm, tools)

print("✅ LangChain Stock Agent initialized successfully!")
print("=" * 70)
print("Available Tools:")
print("=" * 70)
print("1. Search Indian Stocks - Find NSE symbols by company name")
print("2. Scrape Web Info - Gather stock data from web pages")
print("3. Fetch Historical Data - Get price history from Yahoo Finance")
print("4. Plot Graphs - Visualize stock price trends")
print("5. Compare Stocks - Compare multiple stocks side-by-side")
print("=" * 70)
print("\nExample Queries:")
print("=" * 70)
print("- 'What is the performance of TCS stock over 3 months?'")
print("- 'Show me Reliance stock price trends for the last year'")
print("- 'Compare Infosys and Wipro stocks'")
print("- 'Plot a chart for HDFC Bank stock'")
print("=" * 70)


✅ LangChain Stock Agent initialized successfully!
Available Tools:
1. Search Indian Stocks - Find NSE symbols by company name
2. Scrape Web Info - Gather stock data from web pages
3. Fetch Historical Data - Get price history from Yahoo Finance
4. Plot Graphs - Visualize stock price trends
5. Compare Stocks - Compare multiple stocks side-by-side

Example Queries:
- 'What is the performance of TCS stock over 3 months?'
- 'Show me Reliance stock price trends for the last year'
- 'Compare Infosys and Wipro stocks'
- 'Plot a chart for HDFC Bank stock'


In [37]:
# LLM Agent wrapper - Dynamically finds and analyzes Indian stocks with web scraping
def query_stock_agent(query: str, include_web_scraping: bool = True):
    """
    Query the stock agent with natural language.
    Dynamically searches for Indian stocks, fetches their data, 
    scrapes web information, and provides AI-powered analysis.
    """
    from langchain_core.messages import HumanMessage
    import re
    
    print(f"\n🔍 Query: {query}\n")
    print("=" * 70)
    print("⏳ Processing with AI Agent...\n")
    
    try:
        lower_query = query.lower()
        
        # Step 1: Use LLM to extract stock names and determine intent
        print(f"🧠 Analyzing query with Mistral...\n")
        
        extraction_prompt = f"""Extract information from this stock query:

Query: {query}

Respond with:
1. Stock names/companies mentioned (one per line)
2. What the user wants (info/graph/compare/analysis)
3. Time period if mentioned (default: 3mo)

Format:
STOCKS: [comma-separated company names]
INTENT: [what user wants]
PERIOD: [time period or 3mo]"""
        
        extraction_result = llm.invoke([HumanMessage(content=extraction_prompt)])
        print(f"📋 Analysis:\n{extraction_result.content}\n")
        
        # Step 2: Parse the LLM response to extract stocks
        response_text = extraction_result.content.lower()
        
        # Extract stocks section
        stocks_match = re.search(r'stocks:\s*([^\n]+)', response_text)
        if stocks_match:
            stock_names_str = stocks_match.group(1).strip()
            potential_stocks = [s.strip() for s in stock_names_str.split(',')]
        else:
            potential_stocks = []
        
        # Extract period
        period = "3mo"  # default
        period_match = re.search(r'period:\s*(\w+)', response_text)
        if period_match:
            period_str = period_match.group(1).lower()
            period_map = {
                '1d': '1d', '5d': '5d', '1mo': '1mo', '3mo': '3mo',
                '6mo': '6mo', '1y': '1y', '2y': '2y', '5y': '5y',
                'day': '1d', 'week': '5d', 'month': '1mo', 'quarter': '3mo',
                '6': '6mo', 'year': '1y'
            }
            for key, val in period_map.items():
                if key in period_str:
                    period = val
                    break
        
        if not potential_stocks:
            print("❌ No stock names found in query\n")
            print("💡 Try mentioning company names like: TCS, Infosys, Reliance, HDFC, Wipro, etc.\n")
            print(f"{'='*70}\n")
            return
        
        # Step 3: Search for each stock and get NSE symbols
        print(f"🔍 Searching for stocks: {', '.join(potential_stocks)}\n")
        found_stocks = {}
        
        for stock_name in potential_stocks:
            if not stock_name or len(stock_name) < 2:
                continue
            
            print(f"   Searching for '{stock_name}'...")
            search_result = search_indian_stock.func(stock_name)
            
            if search_result.get('success'):
                symbol = search_result.get('symbol')
                company = search_result.get('company_name')
                found_stocks[symbol] = {
                    'symbol': symbol,
                    'company': company,
                    'query_name': stock_name
                }
                print(f"   ✅ Found: {company} ({symbol})\n")
            else:
                print(f"   ⚠️  Not found: {search_result.get('error', 'Unknown error')}\n")
        
        if not found_stocks:
            print("❌ No matching stocks found in database\n")
            print("Available Indian stocks include: TCS, Infosys, Wipro, Reliance, HDFC Bank, ICICI Bank, SBI, etc.\n")
            print(f"{'='*70}\n")
            return
        
        # Step 4: Fetch historical data for each found stock
        print(f"📊 Fetching data for: {', '.join([v['company'] for v in found_stocks.values()])} ({period})\n")
        print("-" * 70 + "\n")
        
        stock_data_results = {}
        
        for symbol, stock_info in found_stocks.items():
            company = stock_info['company']
            
            try:
                result = fetch_stock_historical_data.func(symbol, period)
                
                if 'error' not in result:
                    stock_data_results[symbol] = result
                    
                    print(f"✅ {company} ({symbol})")
                    print(f"   Latest Price: ₹{result['latest_close']:.2f}")
                    print(f"   Open: ₹{result['latest_open']:.2f}")
                    print(f"   Day High: ₹{result['latest_high']:.2f}")
                    print(f"   Day Low: ₹{result['latest_low']:.2f}")
                    print(f"   Volume: {result['avg_volume']:,.0f} (avg)")
                    print(f"   Change: {result['percent_change']:+.2f}% ({result['price_change']:+.2f})")
                    print(f"   52-Week Range: {period}")
                    print(f"   Data Points: {result['data_points']}\n")
                else:
                    print(f"❌ {company}: {result.get('error', 'Unknown error')}\n")
            except Exception as e:
                print(f"❌ Error fetching {company}: {str(e)}\n")
        
        # Step 5: Scrape web information if enabled
        web_info_results = {}
        if include_web_scraping and stock_data_results:
            print("-" * 70 + "\n")
            print("🌐 Scraping web information for detailed analysis...\n")
            
            for symbol, stock_info in found_stocks.items():
                company = stock_info['company']
                if symbol in stock_data_results:
                    try:
                        print(f"   {company}:")
                        web_info = scrape_stock_web_info.func(company, symbol, max_pages=5)
                        web_info_results[symbol] = web_info
                        print(f"   ✓ Scraped {web_info.get('pages_scraped', 0)} pages\n")
                    except Exception as e:
                        print(f"   ⚠️  Error scraping {company}: {str(e)}\n")
        
        # Step 6: Generate AI-powered summary and analysis
        if stock_data_results:
            print("-" * 70 + "\n")
            print("🔮 AI Analysis & Summary:\n")
            
            # Combine all data for analysis
            analysis_data = {
                'stocks': stock_data_results,
                'web_info': web_info_results
            }
            
            summary_prompt = f"""Based on the following stock data and web information, provide a comprehensive analysis:

STOCK DATA:
{json.dumps(stock_data_results, indent=2, default=str)[:2000]}

WEB INFORMATION:
{json.dumps({k: v.get('content', [])[:2] for k, v in web_info_results.items()}, indent=2, default=str)[:1000]}

Please provide:
1. **Stock Performance Summary** - Key metrics and performance indicators
2. **Market Sentiment** - Based on web information gathered
3. **Key Insights** - Important trends or patterns
4. **Investment Outlook** - Brief outlook based on available data
5. **Recommendations** - Any notable observations or cautions

Keep the analysis concise but informative."""
            
            try:
                analysis_result = llm.invoke([HumanMessage(content=summary_prompt)])
                print(f"{analysis_result.content}\n")
            except Exception as e:
                print(f"⚠️  Error generating analysis: {str(e)}\n")
        
        # Step 7: Offer to plot graphs if multiple data points
        if len(stock_data_results) >= 1:
            print("-" * 70 + "\n")
            print("📈 Chart Available:")
            for symbol, data in stock_data_results.items():
                company = found_stocks[symbol]['company']
                if data.get('data_points', 0) > 1:
                    print(f"   • {company} ({symbol}) - Run: plot_stock_graph('{symbol}', '{period}')")
        
        print(f"\n{'='*70}\n")
        
    except Exception as e:
        print(f"❌ Error: {str(e)}\n")
        import traceback
        traceback.print_exc()
        print(f"{'='*70}\n")


In [40]:
# Example 1: Search for a stock by company name
print("\n📊 Example 1: Search for Tata Consultancy Services (TCS) stock\n")
query_stock_agent("What is the current price and performance of Wipro stock over the last 6 months?")


📊 Example 1: Search for Tata Consultancy Services (TCS) stock


🔍 Query: What is the current price and performance of Wipro stock over the last 6 months?

⏳ Processing with AI Agent...

🧠 Analyzing query with Mistral...

📋 Analysis:
 STOCKS: Wipro
INTENT: Current price and performance analysis
PERIOD: Last 6 months (default: 3mo)

🔍 Searching for stocks: wipro

   Searching for 'wipro'...
   ✅ Found: Wipro (WIPRO.NS)

📊 Fetching data for: Wipro (3mo)

----------------------------------------------------------------------

✅ Wipro (WIPRO.NS)
   Latest Price: ₹215.66
   Open: ₹213.26
   Day High: ₹219.39
   Day Low: ₹212.26
   Volume: 11,059,724 (avg)
   Change: -9.35% (-22.25)
   52-Week Range: 3mo
   Data Points: 65

----------------------------------------------------------------------

🌐 Scraping web information for detailed analysis...

   Wipro:
   🌐 Searching for: Wipro stock price news
      ⚠️  Network error on https://business.ndtv.com/search?query=Wipro: ConnectionError
     

In [39]:
# Example 2: Natural language query with company name
print("\n📈 Example 2: Real-time search for Reliance Industries\n")
query_stock_agent("Show me the performance of Reliance Industries over the past year")


📈 Example 2: Real-time search for Reliance Industries


🔍 Query: Show me the performance of Reliance Industries over the past year

⏳ Processing with AI Agent...

🧠 Analyzing query with Mistral...

📋 Analysis:
 STOCKS: Reliance Industries
INTENT: Performance analysis over the past year
PERIOD: Past 12 months (default: 3mo)

🔍 Searching for stocks: reliance industries

   Searching for 'reliance industries'...
   ✅ Found: Reliance (RELIANCE.NS)

📊 Fetching data for: Reliance (3mo)

----------------------------------------------------------------------

✅ Reliance (RELIANCE.NS)
   Latest Price: ₹1424.60
   Open: ₹1431.10
   Day High: ₹1431.80
   Day Low: ₹1418.60
   Volume: 10,812,312 (avg)
   Change: -6.17% (-93.70)
   52-Week Range: 3mo
   Data Points: 65

----------------------------------------------------------------------

🌐 Scraping web information for detailed analysis...

   Reliance:
   🌐 Searching for: Reliance stock price news
      ⚠️  Network error on https://business.ndtv

In [21]:
# Example 3: Compare multiple Indian stocks dynamically
print("\n🔄 Example 3: Comparing Indian IT stocks (Infosys, Wipro, HCL Tech)\n")
query_stock_agent("Compare the performance of Infosys, Wipro, and HCL Technologies over the last 3 months")


🔄 Example 3: Comparing Indian IT stocks (Infosys, Wipro, HCL Tech)


🔍 Query: Compare the performance of Infosys, Wipro, and HCL Technologies over the last 3 months

⏳ Processing with AI Agent...

🧠 Analyzing query with Mistral...

📋 Analysis:
 STOCKS:
1. Infosys
2. Wipro
3. HCL Technologies

INTENT: Compare performance

PERIOD: Last 3 months (default)

🔍 Searching for stocks: 1. infosys

   Searching for '1. infosys'...
   ✅ Found: Infosys (INFY.NS)

📊 Fetching data for: Infosys (3mo)

----------------------------------------------------------------------

✅ Infosys (INFY.NS)
   Latest Price: ₹1421.50
   Open: ₹1370.00
   Day High: ₹1431.00
   Day Low: ₹1367.00
   Volume: 9,083,251 (avg)
   Change: -5.71% (-86.10)
   52-Week Range: 3mo
   Data Points: 65

----------------------------------------------------------------------

🔮 AI Analysis:

 Based on the provided data, here are three key insights about the stock performance and trends:

1. The stock price has been relatively stable 

In [28]:
# Example 4: Interactive query with full web scraping and AI analysis
print("\n💬 Example 4: Complete analysis with web scraping\n")
query_stock_agent("Give me a comprehensive analysis of HDFC Bank stock performance and market insights over 3 months", include_web_scraping=True)



💬 Example 4: Complete analysis with web scraping


🔍 Query: Give me a comprehensive analysis of HDFC Bank stock performance and market insights over 3 months

⏳ Processing with AI Agent...

🧠 Analyzing query with Mistral...

📋 Analysis:
 STOCKS: HDFC Bank
INTENT: Comprehensive analysis (info/graph/compare)
PERIOD: 3 months

🔍 Searching for stocks: hdfc bank

   Searching for 'hdfc bank'...
   ✅ Found: Hdfc Bank (HDFCBANK.NS)

📊 Fetching data for: Hdfc Bank (3mo)

----------------------------------------------------------------------

✅ Hdfc Bank (HDFCBANK.NS)
   Latest Price: ₹925.50
   Open: ₹923.00
   Day High: ₹927.70
   Day Low: ₹918.60
   Volume: 23,978,167 (avg)
   Change: -7.13% (-71.05)
   52-Week Range: 3mo
   Data Points: 65

----------------------------------------------------------------------

🌐 Scraping web information for detailed analysis...

   Hdfc Bank:
   🌐 Searching for: Hdfc Bank stock price news
      ⏱️  Timeout: https://www.cnbc.com/search/?query=Hdfc%20Bank
 

## AI-Powered Stock Analysis with Web Scraping

### 🎯 Key Features:
✅ **Dynamic Stock Search** - Search for any Indian company by name  
✅ **Real-Time Lookup** - Automatically finds NSE symbols  
✅ **Historical Data** - Fetches price history from Yahoo Finance  
✅ **Web Scraping** - Gathers information from financial websites  
✅ **AI Analysis** - Comprehensive LLM-powered analysis and insights  
✅ **Market Sentiment** - Aggregated insights from multiple sources  
✅ **Flexible Queries** - Understands natural language perfectly

### 🌐 Web Scraping & AI Analysis Pipeline:

1. **Extract Company Name** - LLM parses your query
2. **Search Database** - Finds NSE symbol automatically
3. **Fetch Price Data** - Gets historical prices from Yahoo Finance
4. **Scrape Web** - Gathers info from financial websites (Google, CNBC, NDTV, ET)
5. **AI Analysis** - Mistral LLM creates comprehensive summary
6. **Generate Report** - Provides structured insights

### 📊 Supported Indian Stocks:
The agent supports 50+ Indian stocks including:
- **IT Sector**: TCS, Infosys, Wipro, HCL Tech, Tech Mahindra
- **Banks**: HDFC Bank, ICICI Bank, SBI, Axis Bank, Kotak Bank
- **Energy**: Reliance, ONGC, Power Grid
- **Auto**: Maruti, Hero, Bajaj, TVS
- **Pharma**: Cipla, Dr. Reddy's, Sun Pharma, Lupin
- **FMCG**: HUL, Nestlé, Britannia, ITC
- And many more...

### 💻 How It Works:

**Step 1: Natural Language Query**
```python
query_stock_agent("What is the performance of TCS stock over 6 months?")
```

**Step 2: Agent Processing**
- Analyzes your query with Mistral LLM
- Extracts company names and time period
- Searches for matching Indian stocks

**Step 3: Data Collection**
- Fetches historical price data
- Scrapes web pages for market information
- Aggregates multiple data sources

**Step 4: AI Analysis**
- Generates comprehensive report
- Provides market sentiment
- Offers investment insights

### 📝 Example Queries:

**Single Stock Analysis:**
```python
query_stock_agent("What is the price of TCS stock?")
query_stock_agent("Show me Reliance Industries performance over 1 year")
query_stock_agent("How is HDFC Bank doing in the market?")
```

**With Time Period:**
```python
query_stock_agent("Show me TCS stock trend for 6 months")
query_stock_agent("Analyze Infosys stock over the last year")
```

**With Detailed Analysis:**
```python
query_stock_agent("Give me a comprehensive analysis of SBI stock performance and market insights")
query_stock_agent("Provide detailed market sentiment for Wipro stock")
```

**Multiple Stocks:**
```python
query_stock_agent("Compare Infosys and Wipro stocks")
query_stock_agent("Which is better: ICICI or HDFC Bank?")
```

### 📈 Analysis Includes:

1. **Stock Performance Summary** - Price, volume, change metrics
2. **Market Sentiment** - Aggregated from web sources
3. **Key Insights** - Trends and patterns
4. **Investment Outlook** - Brief forecast
5. **Recommendations** - Notable observations

### 🛠️ Advanced Usage:

**Plot a graph:**
```python
plot_stock_graph('TCS.NS', period='6mo', include_volume=True)
```

**Compare multiple stocks:**
```python
query_stock_agent("Compare TCS, Infosys, and Wipro performance")
```

**Disable web scraping (faster):**
```python
query_stock_agent("Show TCS stock data", include_web_scraping=False)
```

### ⚠️ Important Notes:
- The agent uses **Ollama's Mistral** model for intelligent parsing
- Data is fetched from **Yahoo Finance** (reliable historical data)
- Web scraping gathers info from **Google, CNBC, NDTV, ET** and more
- Stock symbols are automatically converted to **NSE format** (e.g., TCS.NS)
- Ensure **Ollama** is running on localhost:11434
- All graphs are saved with timestamps
- Analysis includes both quantitative metrics and market sentiment
